In [94]:
# ── SECTION 0: Imports ──
import pandas as pd
import numpy as np
import re
import warnings

In [95]:
change_log = []

def log_change(idx, name, field, old_val, new_val, source):
    change_log.append({
        'row_index': idx, 'name': name, 'field': field,
        'old_value': old_val, 'new_value': new_val, 'source': source
    })

# 1. LOAD & INITIAL MERGES

In [96]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1: LOAD & INITIAL MERGES
# (This is Notebook 2, but we DON'T save an intermediate file)
# ══════════════════════════════════════════════════════════════════════════════

df = pd.read_csv('./outputs/Global_Graphite_Projects.csv')
print(f"Loaded: {len(df)} rows, {len(df.columns)} columns")

# ── 1.1 Merge Name columns ──
df['Name'] = df['Facility Name'].fillna(df['Site Name']).fillna(df['Deposit Name'])

# ── 1.2 Merge Commodity ──
df['Commodity'] = df['Commodity'].fillna(df['Commodity 1'])

# ── 1.3 Standardise capacity units to metric tons ──
unit_to_mt = {
    'mt': 1, 'Metric tons': 1,
    'kmt': 1_000, 'Thousand metric tons': 1_000, 'tmt': 1_000,
}
mask = df['Capacity Units'].notna() & df['Annual Production Capacity'].notna()
df.loc[mask, 'Annual Production Capacity'] = df.loc[mask].apply(
    lambda r: r['Annual Production Capacity'] * unit_to_mt.get(r['Capacity Units'], 1), axis=1
)
df.loc[mask, 'Capacity Units'] = 'mt'

# ── 1.4 Merge Status columns ──
df['Status'] = (df['Facility Status']
    .fillna(df['Exploration Status'])
    .fillna(df['Operating Status'])
    .fillna(df['Development Status']))

# ── 1.5 Merge Reference Year ──
df['Reference Year'] = df['Year'].fillna(df['Source Year']).fillna(df['Project Active'])

# ── 1.6 Merge ADM1 Name / AMD1 Name ──
df['ADM1 Name'] = df['ADM1 Name'].fillna(df.get('AMD1 Name', pd.Series()))

# ── 1.7 Merge Owner ──
df['Owner'] = df['Equity Owner 1'].fillna(df['Owner Name'])

# ── 1.8 Merge Feature Type ──
if 'Facility Type' in df.columns and 'Feature Type' in df.columns:
    df['Feature Type'] = df['Feature Type'].fillna(df['Facility Type'])
elif 'Facility Type' in df.columns:
    df['Feature Type'] = df['Facility Type']

print(f"After initial merges: {len(df)} rows, {len(df.columns)} columns")

Loaded: 183 rows, 90 columns
After initial merges: 183 rows, 94 columns


# 2. Data Imputation

## 2.1. China - USGS Minerals Yearbook 2023 China advance release

In [97]:
# ══════════════════════════════════════════════════════════════════════════════
# SOURCE: USGS Minerals Yearbook 2023 - China [Advance Release], Feb 2026
# https://pubs.usgs.gov/myb/vol3/2023/myb3-2023-china.pdf
#
# Table 2 (p. 9.24): "Structure of the Mineral Industry in 2023"
#   - Lists 8 graphite facilities with capacities in THOUSAND metric tons
#   - This is the most current publicly available facility-level data for China
#
# Narrative text (p. 9.5–9.6): "Graphite" section under "Industrial Minerals"
#   - Names 3 additional large-scale facilities that came online in 2023
#   - These are NOT listed in Table 2 (they are new capacity additions)
#   - China's total natural graphite capacity reached 1.8 Mt/yr in 2023
#     (up 600,000 t/yr from 2022)
#   - Natural graphite production: 1.21 Mt in 2023 (up from 1.16 Mt in 2022)
#
# IMPORTANT: We focus only on NATURAL graphite. BTR also added 200,000 t/yr
# synthetic capacity - we exclude that.
# ══════════════════════════════════════════════════════════════════════════════

SOURCE = 'USGS Minerals Yearbook 2023 — China (Feb 2026), Table 2 (p. 9.24) & Graphite text (p. 9.5-9.6)'

In [98]:
# Update existing records from Table 2
# Match our existing China facilities to Table 2 entries and update capacities

updates = [
    # (search_keyword_in_Name_or_Operator, new_capacity_mt, new_operator, notes)
    
    # "Mine in Jixi" / Jixi Aoyu: our GDB has 60,000 (from 2019 MYB); Table 2 shows 100 kmt
    ('Aoyu',    100_000, 'Jixi Aoyu Graphite Co. Ltd.',
     'Updated from 60,000 (2019 MYB) to 100,000 (2023 MYB Table 2)'),
    
    # "Plant in Qingdao" / Hensen: our GDB has -999; Table 2 shows 30 kmt
    # Note: Our GDB labels this as "Qingdao Hensen" but Table 2 shows
    # Hensen is in Heilongjiang/Jiangsu/Nei Mongol, not Qingdao
    ('Hensen',  30_000,  'Hensen Graphite Co. Ltd.',
     'Was -999 (unknown); Table 2 shows 30 kmt. Location is Heilongjiang/Jiangsu/Nei Mongol, not Qingdao'),
    
    # "Liumao Mine": our GDB has 80,000; Table 2 (2023) now lists this as
    # "Jixi Puchen Graphite Co. Ltd." (same location, same capacity)
    ('Liumao',  80_000,  'Jixi Puchen Graphite Co. Ltd.',
     'Company renamed from Jixi Liumao (2022 MYB) to Jixi Puchen (2023 MYB). Capacity unchanged at 80 kmt'),
    
    # "Mine in Xinghe": capacity unchanged at 10,000
    ('Xinghe',  10_000,  'Nei Mongol Xinghe Jingyin Graphite Co. Ltd.',
     'Company name spelling updated: Jingxin (2022) → Jingyin (2023). Capacity unchanged'),
    
    # "Qingdao Mine" / Yanxin: capacity unchanged at 28,000
    ('Yanxin',  28_000,  'Qingdao Yanxin Graphite Products Co. Ltd.',
     'Capacity confirmed at 28 kmt in 2023 MYB Table 2'),
]

for keyword, new_cap, new_operator, notes in updates:
    mask = (
        (df['Country (Short Form)'] == 'China') &
        (df['Major Operating Company'].str.contains(keyword, case=False, na=False) |
         df['Facility Name'].str.contains(keyword, case=False, na=False))
    )
    
    matched = df[mask]
    if matched.empty:
        print(f"  ⚠ No match found for '{keyword}' — will add as new record")
        continue
    
    for idx in matched.index:
        old_cap = df.loc[idx, 'Annual Production Capacity']
        old_op = df.loc[idx, 'Major Operating Company']
        
        # Update capacity
        if pd.isna(old_cap) or old_cap < 0 or old_cap != new_cap:
            df.loc[idx, 'Annual Production Capacity'] = new_cap
            df.loc[idx, 'Capacity Units'] = 'mt'
            log_change(idx, df.loc[idx, 'Facility Name'], 'Annual Production Capacity',
                       old_cap, new_cap, SOURCE)
            print(f"  ✓ {df.loc[idx, 'Facility Name']}: capacity {old_cap} → {new_cap} mt")
        else:
            print(f"  - {df.loc[idx, 'Facility Name']}: capacity already correct ({new_cap} mt)")
        
        # Update operator name
        if new_operator and old_op != new_operator:
            df.loc[idx, 'Major Operating Company'] = new_operator
            log_change(idx, df.loc[idx, 'Facility Name'], 'Major Operating Company',
                       old_op, new_operator, SOURCE)
        
        # Update reference year
        df.loc[idx, 'Reference Year'] = 2023

print(f"\nUpdated {len([c for c in change_log if 'Capacity' in c['field']])} capacity values")

  ✓ Mine in Jixi: capacity 60000.0 → 100000 mt
  ✓ Plant in Qingdao: capacity -999.0 → 30000 mt
  - Liumao Mine: capacity already correct (80000 mt)
  - Mine in Xinghe: capacity already correct (10000 mt)
  - Qingdao Mine: capacity already correct (28000 mt)

Updated 2 capacity values


In [99]:
# Add new facilities from Table 2 not in our data
# These are in the official Table 2 but missing from our GDB

table2_new = [
    # (Name, Company, Location, Capacity_mt_or_None, Notes)
    ('Qingdao Haida Graphite',
     'Qingdao Haida Graphite Co. Ltd.',
     'Shandong, Qingdao, Pingdu',
     60_000,
     'Table 2 entry; 60 kmt capacity'),
    
    ('Heilongjiang Hegang Graphite Industry',
     'Heilongjiang Hegang Graphite Industry Co. Ltd.',
     'Heilongjiang, Luobei',
     None,  # NA in Table 2
     'Table 2 entry; capacity NA'),
    
    ('Luobei Haida Graphite',
     'Luobei Haida Graphite Co. Ltd.',
     'Heilongjiang, Hegang',
     None,  # NA in Table 2
     'Table 2 entry; capacity NA'),
]

added_t2 = 0
for name, company, location, cap, notes in table2_new:
    # Check not already present
    keyword = name.split()[0]
    already = df[
        (df['Country (Short Form)'] == 'China') &
        (df['Facility Name'].str.contains(keyword, case=False, na=False) |
         df['Major Operating Company'].str.contains(keyword, case=False, na=False))
    ]
    
    if not already.empty:
        print(f"  - '{name}' already matched by '{already.iloc[0]['Facility Name']}' — skipping")
        continue
    
    new_row = {
        'Region': 'China',
        'Source_Layer': 'CHN_MYB_2023_Table2',
        'Country (Short Form)': 'China',
        'Facility Name': name,
        'Commodity': 'Graphite',
        'Status': 'Assumed Active',
        'Facility Type': 'Mines and Quarries',
        'Annual Production Capacity': cap,
        'Capacity Units': 'mt' if cap else None,
        'Major Operating Company': company,
        'Location Description': location,
        'Reference Year': 2023,
        'Facility Comments': notes,
        'Data Source': SOURCE,
    }
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    log_change(len(df)-1, name, 'NEW RECORD', None, name, SOURCE)
    added_t2 += 1
    print(f"  + Added: {name} ({cap if cap else 'NA'} mt)")

print(f"\nAdded {added_t2} new facilities from Table 2")

  - 'Qingdao Haida Graphite' already matched by 'Plant in Qingdao' — skipping
  + Added: Heilongjiang Hegang Graphite Industry (NA mt)
  + Added: Luobei Haida Graphite (NA mt)

Added 2 new facilities from Table 2


C:\Users\157412\AppData\Local\Temp\ipykernel_27552\1177837915.py:55: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
C:\Users\157412\AppData\Local\Temp\ipykernel_27552\1177837915.py:55: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)


In [100]:
# Add new large facilities from narrative text (p. 9.5–9.6)
# These are major 2023 capacity additions described in the Graphite section
# but NOT listed as individual entries in Table 2

narrative_new = [
    # (Name, Company, Location, Capacity_mt, Stage, Notes)
    ('Hegang Natural Graphite Processing Facility',
     'China Minmetals Co. Ltd.',
     'Heilongjiang, Hegang',
     200_000,
     'Production',
     'Pilot production started late August 2023. '
     '200,000 t/yr nameplate capacity for natural graphite processing.'),
    
    ('BTR Heilongjiang Natural Graphite (Phase 1)',
     'BTR New Material Group Co. Ltd.',
     'Heilongjiang',
     50_000,
     'Production',
     'Phase 1: 50,000 t/yr natural graphite. '
     'Target: 200,000 t/yr natural + 400,000 t/yr synthetic (synthetic excluded). '
     'Only natural graphite capacity counted here.'),
    
    ('Jixi Northeast Graphite Flake Plant',
     'Jixi Northeast Mineral Resources Co. Ltd.',
     'Heilongjiang, Jixi',
     100_000,
     'Construction',
     'Construction of 100,000 t/yr graphite flake plant completed in 2023.'),
]

SOURCE_NARRATIVE = SOURCE.replace('Table 2 (p. 9.24) &', '')  # Just the narrative ref

added_narr = 0
for name, company, location, cap, stage, notes in narrative_new:
    # Check not already present using first distinctive word of company name
    company_kw = company.split()[0]  # 'China', 'BTR', 'Jixi'
    # For "China Minmetals" use "Minmetals"; for "Jixi Northeast" use "Northeast"
    if company_kw in ('China', 'Jixi'):
        company_kw = company.split()[1]
    
    already = df[
        (df['Country (Short Form)'] == 'China') &
        (df['Facility Name'].str.contains(company_kw, case=False, na=False) |
         df['Major Operating Company'].str.contains(company_kw, case=False, na=False))
    ]
    
    if not already.empty:
        print(f"  - '{name}' may overlap with '{already.iloc[0]['Facility Name']}' — skipping")
        continue
    
    new_row = {
        'Region': 'China',
        'Source_Layer': 'CHN_MYB_2023_Narrative',
        'Country (Short Form)': 'China',
        'Facility Name': name,
        'Commodity': 'Graphite',
        'Status': stage,
        'Facility Type': 'Mineral Processing Plants',
        'Annual Production Capacity': cap,
        'Capacity Units': 'mt',
        'Major Operating Company': company,
        'Location Description': location,
        'Reference Year': 2023,
        'Facility Comments': notes,
        'Data Source': SOURCE_NARRATIVE,
    }
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    log_change(len(df)-1, name, 'NEW RECORD', None, name, SOURCE_NARRATIVE)
    added_narr += 1
    print(f"  + Added: {name} ({cap:,} mt)")

print(f"\nAdded {added_narr} new facilities from narrative text")

  + Added: Hegang Natural Graphite Processing Facility (200,000 mt)
  + Added: BTR Heilongjiang Natural Graphite (Phase 1) (50,000 mt)
  + Added: Jixi Northeast Graphite Flake Plant (100,000 mt)

Added 3 new facilities from narrative text


In [101]:
df[df['Region'] == 'China']['Annual Production Capacity'].sum()

np.float64(598000.0)

In [102]:
# ── SUMMARY ──
change_df = pd.DataFrame(change_log)

print(f"\n{'='*70}")
print(f"CHINA DATA IMPUTATION SUMMARY")
print(f"{'='*70}")
print(f"Source: {SOURCE}")
print(f"Total field-level changes: {len(change_df)}")

# Show capacity changes
cap_changes = change_df[change_df['field'] == 'Annual Production Capacity']
if not cap_changes.empty:
    print(f"\nCapacity updates ({len(cap_changes)}):")
    for _, ch in cap_changes.iterrows():
        print(f"  {ch['name']}: {ch['old_value']} → {ch['new_value']} mt")

# Show new records
new_records = change_df[change_df['field'] == 'NEW RECORD']
if not new_records.empty:
    print(f"\nNew records added ({len(new_records)}):")
    for _, ch in new_records.iterrows():
        print(f"  {ch['new_value']}")

# China coverage summary
china = df[(df['Country (Short Form)'] == 'China') & (df['Annual Production Capacity'] > 0)]
total_cap = china['Annual Production Capacity'].sum()
print(f"\n--- China Coverage ---")
print(f"Named facilities with capacity: {len(china)}")
print(f"Total named capacity: {total_cap:,.0f} mt")
print(f"USGS national capacity (2023): 1,400,000 mt/yr")
print(f"Coverage: {100*total_cap/1_400_000:.0f}%")
print(f"Untracked (hundreds of small/medium mines): {1_400_000 - total_cap:,.0f} mt")

# Show all China facilities
print(f"\nAll China graphite facilities:")
china_all = df[df['Country (Short Form)'] == 'China'][
    ['Facility Name', 'Annual Production Capacity', 'Major Operating Company', 'Source_Layer']
]
print(china_all.to_string(index=False))


CHINA DATA IMPUTATION SUMMARY
Source: USGS Minerals Yearbook 2023 — China (Feb 2026), Table 2 (p. 9.24) & Graphite text (p. 9.5-9.6)
Total field-level changes: 12

Capacity updates (2):
  Mine in Jixi: 60000.0 → 100000 mt
  Plant in Qingdao: -999.0 → 30000 mt

New records added (5):
  Heilongjiang Hegang Graphite Industry
  Luobei Haida Graphite
  Hegang Natural Graphite Processing Facility
  BTR Heilongjiang Natural Graphite (Phase 1)
  Jixi Northeast Graphite Flake Plant

--- China Coverage ---
Named facilities with capacity: 8
Total named capacity: 598,000 mt
USGS national capacity (2023): 1,400,000 mt/yr
Coverage: 43%
Untracked (hundreds of small/medium mines): 802,000 mt

All China graphite facilities:
                              Facility Name  Annual Production Capacity                        Major Operating Company            Source_Layer
                                Liumao Mine                     80000.0                  Jixi Puchen Graphite Co. Ltd.  CHN_Mineral_Facilit

In [103]:
# SAVE
df.to_csv('./outputs/Global_Graphite_Projects_imputed.csv', index=False)
print(f"Saved: {len(df)} records to Global_Graphite_Projects_imputed.csv")

change_df.to_csv('./outputs/china_imputation_change_log.csv', index=False)
print(f"Saved: {len(change_df)} changes to china_imputation_change_log.csv")

Saved: 188 records to Global_Graphite_Projects_imputed.csv
Saved: 12 changes to china_imputation_change_log.csv


## 2.2. Australia - CSIRO 2025

In [104]:
# ══════════════════════════════════════════════════════════════════════════════
# SOURCE: "Australian graphite: A path to a global battery market opportunity"
# https://www.csiro.au/en/research/natural-environment/Critical-minerals/Australian-graphite-global-battery-market
# Table A.2: Australian miners active in graphite
#
# This adds resource/reserve data — a new dimension not in our dataset.
# New columns created:
#   - Total Resource (mt)          — Total Mineral Resource in metric tons
#   - Resource Grade (TGC%)        — Total graphitic carbon grade
#   - Contained Graphite (mt)      — Total contained graphite in metric tons
#   - Measured Resource (mt)       — Measured category resource
#   - Measured Grade (TGC%)
#   - Measured Graphite (mt)
#   - Indicated Resource (mt)      — Indicated category resource
#   - Indicated Grade (TGC%)
#   - Indicated Graphite (mt)
#   - Inferred Resource (mt)       — Inferred category resource
#   - Inferred Grade (TGC%)
#   - Inferred Graphite (mt)
#
# All tonnages converted from the PDF's "tonnes" to metric tons (1:1).
# ══════════════════════════════════════════════════════════════════════════════

SOURCE_AUS = 'Australian graphite: A path to a global battery market opportunity, Table A.2'

australia_graphite = pd.DataFrame([
    # (ASX, Company, State, Deposit,
    #  Tot_resource, Tot_grade, Tot_graphite,
    #  Meas_resource, Meas_grade, Meas_graphite,
    #  Ind_resource, Ind_grade, Ind_graphite,
    #  Inf_resource, Inf_grade, Inf_graphite,
    #  ASX_date)
    
    ('KNG', 'Kingsland Minerals', 'NT', 'Leliyn',
     194_600_000, 7.3, 14_200_000,
     None, None, None,
     None, None, None,
     194_600_000, 7.3, 14_200_000,
     '2024-03-13'),
    
    ('RNU', 'Renascor', 'SA', 'Siviour',
     123_600_000, 6.9, 8_500_000,
     16_900_000, 8.6, 1_400_000,
     56_200_000, 6.7, 3_800_000,
     50_500_000, 6.5, 3_300_000,
     '2023-09-14'),
    
    ('IG6', 'International Graphite Ltd', 'WA', 'Springdale',
     49_300_000, 6.43, 3_168_300,
     None, None, None,
     11_500_000, 7.5, 862_500,
     37_800_000, 6.1, 2_305_800,
     '2023-09-12'),
    
    ('ITM', 'Itech Minerals Ltd', 'SA', 'Campoona',
     35_150_000, 5.99, 2_104_000,
     320_000, 12.7, 40_000,
     22_600_000, 5.3, 1_196_000,
     12_230_000, 7.1, 868_000,
     '2024-07-01'),
    
    ('NVX', 'Novonix Ltd', 'QLD', 'Mt Dromedary',
     14_300_000, 13.33, 1_905_700,
     1_000_000, 12.9, 129_000,
     8_500_000, 13.9, 1_181_500,
     4_800_000, 12.4, 595_200,
     '2016-10-20'),
    
    ('GCM', 'Green Critical Minerals Ltd', 'WA', 'McIntosh',
     30_100_000, 4.4, 1_320_000,
     None, None, None,
     19_200_000, 4.44, 850_000,
     10_900_000, 4.33, 470_000,
     '2024-07-01'),
    
    ('LEL', 'Lithium Energy Ltd', 'QLD', 'Bourke',
     9_100_000, 14.4, 1_310_000,
     None, None, None,
     4_500_000, 14.7, 670_000,
     4_500_000, 14.2, 640_000,
     '2023-04-05'),
    
    ('MRC', 'Mineral Commodities', 'WA', 'Munglinup',
     7_990_000, 12.2, 973_190,
     None, None, None,
     4_490_000, 13.1, 588_190,
     3_500_000, 11.0, 385_000,
     '2023-04-28'),
    
    ('LML', 'Lincoln Minerals Ltd', 'SA', 'Kookaburra Gully',
     12_840_000, 7.57, 971_980,
     1_000_000, 11.77, 117_700,
     4_860_000, 8.8, 427_598,
     6_980_000, 6.11, 426_682,
     '2024-04-16'),
    
    ('QGL', 'Quantum Graphite', 'SA', 'Uley',
     6_900_000, 10.98, 757_300,
     800_000, 15.6, 124_800,
     4_200_000, 10.4, 436_800,
     1_900_000, 10.3, 195_700,
     '2023-03-14'),
    
    ('BUX', 'Buxton', 'WA', 'Graphite Bull',
     4_000_000, 16.1, 644_000,
     None, None, None,
     None, None, None,
     4_000_000, 16.1, 644_000,
     '2022-10-12'),
    
    ('OAR', 'Oar Resources', 'SA', 'Oakdale',
     6_320_000, 4.7, 297_040,
     None, None, None,
     2_690_000, 4.7, 126_430,
     3_630_000, 4.7, 170_610,
     '2015-12-02'),
    
    ('LEL', 'Lithium Energy Ltd', 'QLD', 'Corella',
     13_500_000, 9.5, 1_280_000,
     None, None, None,
     None, None, None,
     13_500_000, 9.5, 1_280_000,
     '2023-06-16'),
    
    (None, 'GraphinEx', 'QLD', 'Esmerelda',
     434_500_000, 5.8, 25_350_000,
     None, None, None,
     173_100_000, 5.8, 10_110_000,
     261_400_000, 5.8, 15_240_000,
     None),

], columns=[
    'ASX_Code', 'Company', 'State', 'Deposit',
    'Total_Resource_t', 'Total_Grade_TGC', 'Total_Graphite_t',
    'Measured_Resource_t', 'Measured_Grade_TGC', 'Measured_Graphite_t',
    'Indicated_Resource_t', 'Indicated_Grade_TGC', 'Indicated_Graphite_t',
    'Inferred_Resource_t', 'Inferred_Grade_TGC', 'Inferred_Graphite_t',
    'ASX_Date',
])

print(f"Australian graphite deposits: {len(australia_graphite)} projects")
print(f"Total contained graphite: {australia_graphite['Total_Graphite_t'].sum():,.0f} mt")
australia_graphite[['Company', 'Deposit', 'State', 'Total_Resource_t', 'Total_Grade_TGC', 'Total_Graphite_t']]

Australian graphite deposits: 14 projects
Total contained graphite: 62,781,510 mt


,Company,Deposit,State,Total_Resource_t,Total_Grade_TGC,Total_Graphite_t
0,Kingsland Minerals,Leliyn,NT,194600000,7.30,14200000
1,Renascor,Siviour,SA,123600000,6.90,8500000
2,International Graphite Ltd,Springdale,WA,49300000,6.43,3168300
3,Itech Minerals Ltd,Campoona,SA,35150000,5.99,2104000
4,Novonix Ltd,Mt Dromedary,QLD,14300000,13.33,1905700
5,Green Critical Minerals Ltd,McIntosh,WA,30100000,4.40,1320000
6,Lithium Energy Ltd,Bourke,QLD,9100000,14.40,1310000
7,Mineral Commodities,Munglinup,WA,7990000,12.20,973190
8,Lincoln Minerals Ltd,Kookaburra Gully,SA,12840000,7.57,971980
9,Quantum Graphite,Uley,SA,6900000,10.98,757300


In [105]:
# ── Add Australian deposits to the main dataframe ──
# These are all new — Australia has no records in our current dataset

new_rows = []
for _, row in australia_graphite.iterrows():
    new_rows.append({
        # Core fields
        'Region': 'Australia',
        'Source_Layer': 'AUS_Graphite_Deposits',
        'Country (Short Form)': 'Australia',
        'Name': row['Deposit'],
        'Commodity': 'Graphite',
        'Status': 'Exploration',  # These are resource-stage deposits
        'Deposit Type': 'Natural flake graphite',
        'Major Operating Company': row['Company'],
        'ADM1 Name': row['State'],
        'Data Source': SOURCE_AUS,
        'Reference Year': int(row['ASX_Date'][:4]) if pd.notna(row['ASX_Date']) else None,
        'Facility Comments': f"ASX: {row['ASX_Code']}" if pd.notna(row['ASX_Code']) else None,
        
        # Resource data — new columns
        'Total Resource (mt)': row['Total_Resource_t'],
        'Resource Grade (TGC%)': row['Total_Grade_TGC'],
        'Contained Graphite (mt)': row['Total_Graphite_t'],
        'Measured Resource (mt)': row['Measured_Resource_t'],
        'Measured Grade (TGC%)': row['Measured_Grade_TGC'],
        'Measured Graphite (mt)': row['Measured_Graphite_t'],
        'Indicated Resource (mt)': row['Indicated_Resource_t'],
        'Indicated Grade (TGC%)': row['Indicated_Grade_TGC'],
        'Indicated Graphite (mt)': row['Indicated_Graphite_t'],
        'Inferred Resource (mt)': row['Inferred_Resource_t'],
        'Inferred Grade (TGC%)': row['Inferred_Grade_TGC'],
        'Inferred Graphite (mt)': row['Inferred_Graphite_t'],
    })

new_aus_df = pd.DataFrame(new_rows)
df = pd.concat([df, new_aus_df], ignore_index=True)

for _, row in new_aus_df.iterrows():
    log_change(None, row['Name'], 'NEW RECORD', None, row['Name'], SOURCE_AUS)

print(f"Added {len(new_aus_df)} Australian deposits")
print(f"Dataset now: {len(df)} rows, {len(df.columns)} columns")
print(f"\nNew resource columns added:")
for col in [c for c in df.columns if 'Resource' in c or 'Grade' in c or 'Graphite (mt)' in c]:
    non_null = df[col].notna().sum()
    print(f"  {col}: {non_null} values")

Added 14 Australian deposits
Dataset now: 202 rows, 106 columns

New resource columns added:
  Total Resource (mt): 14 values
  Resource Grade (TGC%): 14 values
  Contained Graphite (mt): 14 values
  Measured Resource (mt): 5 values
  Measured Grade (TGC%): 5 values
  Measured Graphite (mt): 5 values
  Indicated Resource (mt): 11 values
  Indicated Grade (TGC%): 11 values
  Indicated Graphite (mt): 11 values
  Inferred Resource (mt): 14 values
  Inferred Grade (TGC%): 14 values
  Inferred Graphite (mt): 14 values


In [106]:
# ── Summary ──
print(f"\n{'='*70}")
print(f"AUSTRALIA DATA SUMMARY")
print(f"{'='*70}")
print(f"Source: {SOURCE_AUS}")
print(f"Projects added: {len(new_aus_df)}")
print(f"Total mineral resource: {australia_graphite['Total_Resource_t'].sum()/1e6:,.0f} Mt ore")
print(f"Total contained graphite: {australia_graphite['Total_Graphite_t'].sum()/1e6:,.1f} Mt graphite")
print(f"\nTop 5 by contained graphite:")
top5 = australia_graphite.nlargest(5, 'Total_Graphite_t')[
    ['Deposit', 'Company', 'Total_Resource_t', 'Total_Grade_TGC', 'Total_Graphite_t']
].copy()
top5['Total_Resource_t'] = (top5['Total_Resource_t'] / 1e6).round(1)
top5['Total_Graphite_t'] = (top5['Total_Graphite_t'] / 1e6).round(1)
top5.columns = ['Deposit', 'Company', 'Resource (Mt)', 'Grade (TGC%)', 'Graphite (Mt)']
print(top5.to_string(index=False))

print(f"\n[!]  Note: These are MINERAL RESOURCES, not production capacity.")
print(f"   None of these deposits are currently in production.")
print(f"   The resource columns will be NaN for all non-Australian records")
print(f"   until resource data is added for other regions.")


AUSTRALIA DATA SUMMARY
Source: Australian graphite: A path to a global battery market opportunity, Table A.2
Projects added: 14
Total mineral resource: 942 Mt ore
Total contained graphite: 62.8 Mt graphite

Top 5 by contained graphite:
   Deposit                    Company  Resource (Mt)  Grade (TGC%)  Graphite (Mt)
 Esmerelda                  GraphinEx          434.5          5.80           25.4
    Leliyn         Kingsland Minerals          194.6          7.30           14.2
   Siviour                   Renascor          123.6          6.90            8.5
Springdale International Graphite Ltd           49.3          6.43            3.2
  Campoona         Itech Minerals Ltd           35.2          5.99            2.1

[!]  Note: These are MINERAL RESOURCES, not production capacity.
   None of these deposits are currently in production.
   The resource columns will be NaN for all non-Australian records
   until resource data is added for other regions.


In [107]:
# ── Sanity check: Total Resource = Measured + Indicated + Inferred ──
check = australia_graphite[['Deposit', 'Company',
    'Total_Resource_t', 'Measured_Resource_t', 'Indicated_Resource_t', 'Inferred_Resource_t',
    'Total_Graphite_t', 'Measured_Graphite_t', 'Indicated_Graphite_t', 'Inferred_Graphite_t',
]].copy()

# Sum the components (treating NaN as 0)
check['Calc_Resource'] = check[['Measured_Resource_t', 'Indicated_Resource_t', 'Inferred_Resource_t']].sum(axis=1)
check['Calc_Graphite'] = check[['Measured_Graphite_t', 'Indicated_Graphite_t', 'Inferred_Graphite_t']].sum(axis=1)

check['Resource_Diff'] = check['Total_Resource_t'] - check['Calc_Resource']
check['Graphite_Diff'] = check['Total_Graphite_t'] - check['Calc_Graphite']

print("Resource tonnage check (Total - Sum of M+I+I):")
print(check[['Deposit', 'Total_Resource_t', 'Calc_Resource', 'Resource_Diff']].to_string(index=False))

print("\nContained graphite check (Total - Sum of M+I+I):")
print(check[['Deposit', 'Total_Graphite_t', 'Calc_Graphite', 'Graphite_Diff']].to_string(index=False))

# Flag mismatches
mismatches = check[(check['Resource_Diff'].abs() > 1) | (check['Graphite_Diff'].abs() > 1)]
if mismatches.empty:
    print("\n✅ All totals match the sum of Measured + Indicated + Inferred")
else:
    print(f"\n⚠️  {len(mismatches)} mismatches found:")
    print(mismatches[['Deposit', 'Resource_Diff', 'Graphite_Diff']].to_string(index=False))

Resource tonnage check (Total - Sum of M+I+I):
         Deposit  Total_Resource_t  Calc_Resource  Resource_Diff
          Leliyn         194600000    194600000.0            0.0
         Siviour         123600000    123600000.0            0.0
      Springdale          49300000     49300000.0            0.0
        Campoona          35150000     35150000.0            0.0
    Mt Dromedary          14300000     14300000.0            0.0
        McIntosh          30100000     30100000.0            0.0
          Bourke           9100000      9000000.0       100000.0
       Munglinup           7990000      7990000.0            0.0
Kookaburra Gully          12840000     12840000.0            0.0
            Uley           6900000      6900000.0            0.0
   Graphite Bull           4000000      4000000.0            0.0
         Oakdale           6320000      6320000.0            0.0
         Corella          13500000     13500000.0            0.0
       Esmerelda         434500000    43450

## 2.3. USGS Flake Graphite Deposits

In [108]:
# ══════════════════════════════════════════════════════════════════════════════
# SOURCE: USGS Flake Graphite Deposits dataset (2024)
# 72 deposits worldwide with ore tonnage, grade (TGC%), and contained graphite
#
# Matching strategy:
#   1. Fuzzy name match (deposit name keywords) + same country
#   2. If no name match, proximity match (within 50km by lat/lon)
#   3. If matched: update resource columns
#   4. If not matched: add as new record
# ══════════════════════════════════════════════════════════════════════════════

import numpy as np

SOURCE_USGS_FLAKE = 'USGS Flake Graphite Deposits 2024'

# ── Load and clean the USGS CSV ──
usgs_fg = pd.read_csv('./inputs/flake_graphite_dep_usgs_2024.csv', encoding='latin1')
usgs_fg.columns = usgs_fg.columns.str.strip()

# Convert string tonnages to numeric (remove commas)
usgs_fg['Ore_tonnage'] = pd.to_numeric(usgs_fg['Ore_tonnage'].str.replace(',', ''), errors='coerce')
usgs_fg['Graphite_tonnage'] = pd.to_numeric(usgs_fg['Graphite_tonnage'].str.replace(',', ''), errors='coerce')

# Standardise country names to match our dataframe
usgs_fg['Country'] = usgs_fg['Country'].replace({
    'Malawai': 'Malawi',
    'South Korea': 'Korea, South',
    'United States': 'United States',  # not in our df
})

print(f"Loaded USGS flake graphite: {len(usgs_fg)} deposits")
print(f"Countries: {usgs_fg['Country'].nunique()}")

Loaded USGS flake graphite: 72 deposits
Countries: 19


In [109]:
# ── Matching helper functions ──

def haversine_km(lat1, lon1, lat2, lon2):
    """Approximate distance in km between two lat/lon points."""
    R = 6371  # Earth radius in km
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (np.sin(dlat/2)**2 + 
         np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2)
    return R * 2 * np.arcsin(np.sqrt(a))

def name_similarity(name1, name2):
    """Check if two deposit names share significant keywords."""
    if pd.isna(name1) or pd.isna(name2):
        return False
    # Normalise
    stop_words = {'mine', 'project', 'graphite', 'deposit', 'the', 'in', 'of', 'and', 'production'}
    def get_keywords(name):
        words = str(name).lower().replace('-', ' ').replace('(', ' ').replace(')', ' ').split()
        return {w for w in words if w not in stop_words and len(w) > 2}
    
    kw1 = get_keywords(name1)
    kw2 = get_keywords(name2)
    
    if not kw1 or not kw2:
        return False
    
    # Check if any significant keyword matches
    overlap = kw1 & kw2
    return len(overlap) > 0

PROXIMITY_THRESHOLD_KM = 10

In [110]:
# ── Match each USGS deposit against our existing data ──

# Collect all names from our df into a single searchable column
# (handles both pre- and post-merge column structures)
# Always rebuild Name from source columns, filling gaps
name_sources = [df.get('Facility Name', pd.Series()),
                df.get('Site Name', pd.Series()),
                df.get('Deposit Name', pd.Series())]
if 'Name' not in df.columns:
    df['Name'] = pd.Series(dtype='object')
for src in name_sources:
    df['Name'] = df['Name'].fillna(src)

match_results = []  # Track every match decision for review

for usgs_idx, usgs_row in usgs_fg.iterrows():
    usgs_name = usgs_row['Deposit_name']
    usgs_country = usgs_row['Country']
    usgs_lat = usgs_row['Latitude']
    usgs_lon = usgs_row['Longitude']
    
    best_match_idx = None
    match_type = None
    match_distance = None
    
    # ── Pass 1: Name match within same country ──
    country_mask = df['Country (Short Form)'] == usgs_country
    candidates = df[country_mask]
    
    for df_idx, df_row in candidates.iterrows():
        if name_similarity(usgs_name, df_row['Name']):
            best_match_idx = df_idx
            match_type = 'name'
            break
    
    # ── Pass 2: If no name match, try proximity ──
    if best_match_idx is None:
        for df_idx, df_row in candidates.iterrows():
            df_lat = df_row.get('DD Latitude')
            df_lon = df_row.get('DD Longitude')
            if pd.notna(df_lat) and pd.notna(df_lon) and pd.notna(usgs_lat) and pd.notna(usgs_lon):
                dist = haversine_km(usgs_lat, usgs_lon, df_lat, df_lon)
                if dist < PROXIMITY_THRESHOLD_KM:
                    best_match_idx = df_idx
                    match_type = f'proximity ({dist:.1f} km)'
                    match_distance = dist
                    break
    
    match_results.append({
        'usgs_idx': usgs_idx,
        'usgs_name': usgs_name,
        'usgs_country': usgs_country,
        'matched_idx': best_match_idx,
        'matched_name': df.loc[best_match_idx, 'Name'] if best_match_idx is not None else None,
        'match_type': match_type if match_type else 'NO MATCH',
        'ore_tonnage': usgs_row['Ore_tonnage'],
        'grade': usgs_row['Grade_pct_Cg'],
    })

match_df = pd.DataFrame(match_results)

print("=== MATCHING RESULTS ===")
print(f"\nMatched by name:      {(match_df['match_type'] == 'name').sum()}")
print(f"Matched by proximity: {match_df['match_type'].str.startswith('proximity').sum()}")
print(f"No match (new):       {(match_df['match_type'] == 'NO MATCH').sum()}")

print("\n--- Matched records ---")
matched = match_df[match_df['match_type'] != 'NO MATCH']
for _, r in matched.iterrows():
    print(f"  {r['usgs_name']} ({r['usgs_country']}) → {r['matched_name']} [{r['match_type']}]")

print("\n--- New deposits (no match) ---")
unmatched = match_df[match_df['match_type'] == 'NO MATCH']
for _, r in unmatched.iterrows():
    print(f"  {r['usgs_name']} ({r['usgs_country']})")

=== MATCHING RESULTS ===

Matched by name:      25
Matched by proximity: 3
No match (new):       44

--- Matched records ---
  McIntosh (Australia) → McIntosh [name]
  Munglinup Graphite Project (Australia) → Munglinup [name]
  Burke-Mt Dromedary (Australia) → Mt Dromedary [name]
  Beishan (China) → Yunshan [proximity (0.2 km)]
  Liumao Mine (China) → Liumao Mine [name]
  Graphmada Mining Complex (Madagascar) → Graphmada Mine [name]
  Molo Graphite Project (Madagascar) → Molo [name]
  Malingunde (Malawi) → Malingunde [name]
  Lurio Graphite Project (Mozambique) → Dombeya [proximity (7.8 km)]
  Ancuabe (Mozambique) → Ancuabe Graphite Mine [name]
  Balama Central (Mozambique) → Balama Graphite Operation [name]
  Montepuez (Mozambique) → Montepuez central [name]
  Balama (Mozambique) → Balama Graphite Operation [name]
  Nicanda Hill, West & Cobra Plains (Mozambique) → Balama [proximity (0.9 km)]
  Black Range Deposit (Namibia) → Black Range [name]
  Okanjande-Okorusu (Namibia) → Okanjande

In [111]:
# ══════════════════════════════════════════════════════════════════════════════
# REVIEW STEP: Print matches for manual verification before applying changes.
# If any match looks wrong, add it to the exclusion list below.
# ══════════════════════════════════════════════════════════════════════════════

# Add any bad matches here after reviewing the output above
false_matches = [
    # ('usgs_name', 'matched_name'),  # Example: exclude if wrongly matched
]

# Filter out false matches
for usgs_name, matched_name in false_matches:
    mask = (match_df['usgs_name'] == usgs_name) & (match_df['matched_name'] == matched_name)
    match_df.loc[mask, 'match_type'] = 'NO MATCH'
    match_df.loc[mask, 'matched_idx'] = None
    match_df.loc[mask, 'matched_name'] = None
    print(f"  Excluded false match: {usgs_name} → {matched_name}")

In [112]:
# ── STEP 1: Update existing records with resource data ──

updated = 0
for _, m in match_df[match_df['match_type'] != 'NO MATCH'].iterrows():
    idx = m['matched_idx']
    usgs_row = usgs_fg.loc[m['usgs_idx']]
    
    old_resource = df.loc[idx, 'Total Resource (mt)'] if 'Total Resource (mt)' in df.columns else None
    old_grade = df.loc[idx, 'Resource Grade (TGC%)'] if 'Resource Grade (TGC%)' in df.columns else None
    
    # Update resource tonnage
    if pd.isna(old_resource) or old_resource == 0:
        df.loc[idx, 'Total Resource (mt)'] = usgs_row['Ore_tonnage']
        log_change(idx, df.loc[idx, 'Name'], 'Total Resource (mt)',
                   old_resource, usgs_row['Ore_tonnage'], SOURCE_USGS_FLAKE)
    
    # Update grade
    if pd.isna(old_grade) or old_grade == 0:
        df.loc[idx, 'Resource Grade (TGC%)'] = usgs_row['Grade_pct_Cg']
        log_change(idx, df.loc[idx, 'Name'], 'Resource Grade (TGC%)',
                   old_grade, usgs_row['Grade_pct_Cg'], SOURCE_USGS_FLAKE)
    
    # Update contained graphite
    df.loc[idx, 'Contained Graphite (mt)'] = usgs_row['Graphite_tonnage']
    
    # Append source reference
    existing_source = df.loc[idx, 'Data Source'] if pd.notna(df.loc[idx, 'Data Source']) else ''
    if SOURCE_USGS_FLAKE not in str(existing_source):
        new_source = f"{existing_source}; {SOURCE_USGS_FLAKE}; {usgs_row['Reference']}".strip('; ')
        df.loc[idx, 'Data Source'] = new_source
    
    updated += 1

print(f"Updated {updated} existing records with resource data")

Updated 28 existing records with resource data


In [113]:
# ── STEP 2: Add new deposits that didn't match anything ──

new_deposits = match_df[match_df['match_type'] == 'NO MATCH']
added = 0

for _, m in new_deposits.iterrows():
    usgs_row = usgs_fg.loc[m['usgs_idx']]
    
    new_row = {
        'Region': usgs_row['Country'],  # Will need manual region assignment
        'Source_Layer': 'USGS_Flake_Graphite_2024',
        'Country (Short Form)': usgs_row['Country'],
        'Name': usgs_row['Deposit_name'],
        'Commodity': 'Graphite',
        'Commodity Product': 'Flake graphite',
        'Status': 'Exploration',
        'Deposit Type': 'Flake graphite deposit',
        'ADM1 Name': usgs_row['State_Province_Region_County'],
        'DD Latitude': usgs_row['Latitude'],
        'DD Longitude': usgs_row['Longitude'],
        'Data Source': SOURCE_USGS_FLAKE,
        'Data Source': f"{SOURCE_USGS_FLAKE}; {usgs_row['Reference']}",
        
        # Resource data
        'Total Resource (mt)': usgs_row['Ore_tonnage'],
        'Resource Grade (TGC%)': usgs_row['Grade_pct_Cg'],
        'Contained Graphite (mt)': usgs_row['Graphite_tonnage'],
        'Reference Year': int(re.findall(r'\b(20\d{2}|19\d{2})\b', str(usgs_row['Reference']))[-1]) 
                          if re.findall(r'\b(20\d{2}|19\d{2})\b', str(usgs_row['Reference'])) else None,
    }
    
    df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
    log_change(len(df)-1, usgs_row['Deposit_name'], 'NEW RECORD',
               None, usgs_row['Deposit_name'], SOURCE_USGS_FLAKE)
    added += 1

print(f"Added {added} new deposits from USGS flake graphite dataset")

Added 44 new deposits from USGS flake graphite dataset


In [114]:
# ── Assign proper region names for new deposits ──
region_map = {
    'Australia': 'Australia',
    'Canada': 'North America',
    'Norway': 'Europe',
    'Sweden': 'Europe',
    'Greenland': 'Europe',
    'Russia': 'Europe/Asia',
    'Ukraine': 'Europe',
    'United States': 'North America',
    'South Africa': 'Africa',
    'Guinea': 'Africa',
    'Madagascar': 'Africa',
    'Malawi': 'Africa',
    'Mozambique': 'Africa',
    'Namibia': 'Africa',
    'Tanzania': 'Africa',
    'Brazil': 'Latin America & Caribbean',
    'Mexico': 'Latin America & Caribbean',
    'China': 'China',
    'Korea, South': 'SW Asia',
}

mask = df['Source_Layer'] == 'USGS_Flake_Graphite_2024'
df.loc[mask, 'Region'] = df.loc[mask, 'Country (Short Form)'].map(region_map)

print("Region assignments for new deposits:")
print(df.loc[mask, ['Country (Short Form)', 'Region']].drop_duplicates().to_string(index=False))

Region assignments for new deposits:
Country (Short Form)                    Region
           Australia                 Australia
              Brazil Latin America & Caribbean
              Canada             North America
               China                     China
           Greenland                    Europe
              Guinea                    Africa
              Malawi                    Africa
              Mexico Latin America & Caribbean
              Norway                    Europe
              Russia               Europe/Asia
        South Africa                    Africa
              Sweden                    Europe
             Ukraine                    Europe
       United States             North America


In [115]:
# ── Summary ──
print(f"\n{'='*70}")
print(f"USGS FLAKE GRAPHITE INTEGRATION SUMMARY")
print(f"{'='*70}")
print(f"Source: {SOURCE_USGS_FLAKE}")
print(f"USGS deposits processed: {len(usgs_fg)}")
print(f"Matched to existing records: {len(match_df[match_df['match_type'] != 'NO MATCH'])}")
print(f"  - By name: {(match_df['match_type'] == 'name').sum()}")
print(f"  - By proximity: {match_df['match_type'].str.startswith('proximity').sum()}")
print(f"New deposits added: {added}")
print(f"Dataset now: {len(df)} rows, {len(df.columns)} columns")

# Resource coverage
has_resource = df['Total Resource (mt)'].notna().sum()
has_grade = df['Resource Grade (TGC%)'].notna().sum()
print(f"\nResource data coverage:")
print(f"  Records with Total Resource: {has_resource} / {len(df)}")
print(f"  Records with Grade: {has_grade} / {len(df)}")

# Show resource data by country
resource_by_country = df[df['Total Resource (mt)'].notna()].groupby('Country (Short Form)').agg(
    deposits=('Name', 'count'),
    total_resource_mt=('Total Resource (mt)', 'sum'),
    total_graphite_mt=('Contained Graphite (mt)', 'sum'),
    avg_grade=('Resource Grade (TGC%)', 'mean'),
).sort_values('total_graphite_mt', ascending=False)

resource_by_country['total_resource_mt'] = (resource_by_country['total_resource_mt'] / 1e6).round(1)
resource_by_country['total_graphite_mt'] = (resource_by_country['total_graphite_mt'] / 1e6).round(2)
resource_by_country['avg_grade'] = resource_by_country['avg_grade'].round(2)
resource_by_country.columns = ['Deposits', 'Total Resource (Mt)', 'Contained Graphite (Mt)', 'Avg Grade (TGC%)']
print(f"\nResource summary by country:")
print(resource_by_country.to_string())


USGS FLAKE GRAPHITE INTEGRATION SUMMARY
Source: USGS Flake Graphite Deposits 2024
USGS deposits processed: 72
Matched to existing records: 28
  - By name: 25
  - By proximity: 3
New deposits added: 44
Dataset now: 246 rows, 106 columns

Resource data coverage:
  Records with Total Resource: 78 / 246
  Records with Grade: 78 / 246

Resource summary by country:
                      Deposits  Total Resource (Mt)  Contained Graphite (Mt)  Avg Grade (TGC%)
Country (Short Form)                                                                          
Mozambique                   5               1937.1                   331.70              8.45
Australia                   18                975.6                    65.84              9.06
Tanzania                     6               1071.6                    63.82              7.96
China                        3                730.9                    32.62              9.91
Canada                      11                556.8                

In [116]:
df

,Region,Source_Layer,Country (Short Form),Facility Name,Site Name,Deposit Name,Commodity Type,Commodity,Commodity 1,Facility Status,...,Contained Graphite (mt),Measured Resource (mt),Measured Grade (TGC%),Measured Graphite (mt),Indicated Resource (mt),Indicated Grade (TGC%),Indicated Graphite (mt),Inferred Resource (mt),Inferred Grade (TGC%),Inferred Graphite (mt)
0,Africa,AFR_Mineral_Facilities,Madagascar,Gallois Mine,NaN,NaN,Industrial,Graphite,NaN,Assumed Active,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa,AFR_Mineral_Facilities,Madagascar,Graphmada Mine,NaN,NaN,Industrial,Graphite,NaN,Assumed Active,...,2785500.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Africa,AFR_Mineral_Facilities,Madagascar,Mine in Atsinanana Region,NaN,NaN,Industrial,Graphite,NaN,Assumed Active,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Africa,AFR_Mineral_Facilities,Madagascar,Sahamamy-Sahasoa Mine,NaN,NaN,Industrial,Graphite,NaN,Assumed Active,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Africa,AFR_Mineral_Facilities,Mozambique,Ancuabe Graphite Mine,NaN,NaN,Industrial,Graphite,NaN,Assumed Active,...,4584000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
241,Europe,USGS_Flake_Graphite_2024,Ukraine,NaN,NaN,NaN,NaN,Graphite,NaN,NaN,...,5500000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
242,North America,USGS_Flake_Graphite_2024,United States,NaN,NaN,NaN,NaN,Graphite,NaN,NaN,...,4074955.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
243,North America,USGS_Flake_Graphite_2024,United States,NaN,NaN,NaN,NaN,Graphite,NaN,NaN,...,17074518.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
244,Australia,USGS_Flake_Graphite_2024,Australia,NaN,NaN,NaN,NaN,Graphite,NaN,NaN,...,644000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2.4. Europe

### 2.4.1. Norway

In [117]:
# ══════════════════════════════════════════════════════════════════════════════
# SOURCE: Gautneb et al. (2020) "The Graphite Occurrences of Northern Norway,
#         a Review of Geology, Geophysics, and Resources"
#         Minerals 2020, 10, 626; doi:10.3390/min10070626
#         Table 1 (p. 4): 28 graphite occurrences across 3 provinces
#
# Notes:
#   - Tonnages are INFERRED resources estimated from geophysics + sampling
#   - Grade is % TC (Total Carbon), equivalent to our Resource Grade (TGC%)
#   - Tonnages in the paper are in Mt — we convert to metric tons
#   - 10 of these 28 overlap with the USGS flake graphite CSV (already added)
#   - Province info from the paper: Senja (#1–7), Vesterålen (#8–26),
#     Holandsfjorden (#27–28)
# ══════════════════════════════════════════════════════════════════════════════

SOURCE_NORWAY = (
    'Gautneb et al. (2020), Minerals 10(7), 626, Table 1. '
    'doi:10.3390/min10070626'
)

norway_graphite = pd.DataFrame([
    # (No, Name, Province, TC%, Tonnage_Mt, Contained_Graphite_Mt)
    (1,  'Trælen',       'Senja',          31.0,  1.80,  0.56),
    (2,  'Vardfjellet',  'Senja',           9.2, 12.84,  1.18),
    (3,  'Hesten',       'Senja',           5.8,  2.07,  0.12),
    (4,  'Bukken',       'Senja',           6.5, 51.03,  3.34),
    (5,  'Litljkollen',  'Senja',           5.3, 34.54,  1.83),
    (6,  'Grunnvåg',     'Senja',           5.2, 22.77,  1.19),
    (7,  'Skardsvåg',    'Senja',           2.1,  None,  None),
    (8,  'Haugsnes',     'Vesterålen',     16.2,  8.40,  1.36),
    (9,  'Kjerkhaugen',  'Vesterålen',      6.5,  2.42,  0.16),
    (10, 'Møkland',      'Vesterålen',     13.2,  3.40,  0.45),
    (11, 'Rise',         'Vesterålen',      7.9,  0.19,  0.01),
    (12, 'Sommarland',   'Vesterålen',     12.5,  0.85,  0.11),
    (13, 'Brenna',       'Vesterålen',     10.1,  7.94,  0.80),
    (14, 'Evassåsen',    'Vesterålen',      7.6,  2.12,  0.16),
    (15, 'Vikeid Central','Vesterålen',    13.8,  8.89,  1.23),
    (16, 'Vikeid West',  'Vesterålen',     11.3, 29.63,  3.35),
    (17, 'Ånstad',       'Vesterålen',     36.8,  0.21,  0.08),
    (18, 'Alsvåg',       'Vesterålen',      8.9,  0.25,  0.02),
    (19, 'Instøya',      'Vesterålen',      9.3, 14.82,  1.42),
    (20, 'Rødhamran',    'Vesterålen',     14.8,  1.38,  0.20),
    (21, 'Romset',       'Vesterålen',     14.7,  9.63,  1.42),
    (22, 'Skogsøya',     'Vesterålen',     20.0,  1.42,  0.30),
    (23, 'Smines',       'Vesterålen',      7.1, 18.89,  1.34),
    (24, 'Svinøya',      'Vesterålen',     11.7,  0.02,  0.02),
    (25, 'Jennestad',    'Vesterålen',      9.6,  3.66,  0.35),
    (26, 'Myre',         'Vesterålen',     None,  None,  None),
    (27, 'Nord-Værnes',  'Holandsfjorden',  4.1,  0.60,  0.02),
    (28, 'Rendalsvik',   'Holandsfjorden', 11.1,  1.90,  0.21),
], columns=['Occ_No', 'Name', 'Province', 'TC_pct', 'Tonnage_Mt', 'Contained_Graphite_Mt'])

# Convert Mt to metric tons
norway_graphite['Tonnage_t'] = norway_graphite['Tonnage_Mt'] * 1_000_000
norway_graphite['Contained_Graphite_t'] = norway_graphite['Contained_Graphite_Mt'] * 1_000_000

print(f"Norwegian graphite occurrences from Gautneb et al. (2020): {len(norway_graphite)}")
print(f"With tonnage data: {norway_graphite['Tonnage_Mt'].notna().sum()}")
print(f"Total ore tonnage: {norway_graphite['Tonnage_Mt'].sum():.1f} Mt")
print(f"Total contained graphite: {norway_graphite['Contained_Graphite_Mt'].sum():.2f} Mt")

Norwegian graphite occurrences from Gautneb et al. (2020): 28
With tonnage data: 26
Total ore tonnage: 241.7 Mt
Total contained graphite: 21.23 Mt


In [118]:
# ── Match against existing Norway records (from USGS flake graphite CSV) ──
# The USGS CSV already added 10 Norwegian deposits. We match by name keyword.

norway_existing = df[df['Country (Short Form)'] == 'Norway']
print(f"Existing Norway records in dataset: {len(norway_existing)}")
if not norway_existing.empty:
    print("Names:", norway_existing['Name'].tolist())

updated_nor = 0
added_nor = 0

for _, nor_row in norway_graphite.iterrows():
    nor_name = nor_row['Name']
    
    # Try to find existing record by name keyword
    # Handle special chars: Grunnvåg, Møkland, etc.
    keyword = nor_name.split()[0]  # First word of the name
    
    mask = (
        (df['Country (Short Form)'] == 'Norway') &
        (df['Name'].str.contains(keyword, case=False, na=False))
    )
    matched = df[mask]
    
    if not matched.empty:
        # ── Update existing record ──
        idx = matched.index[0]
        existing_name = df.loc[idx, 'Name']
        
        # Update grade if we have a better value from this source
        # (Gautneb uses TC% from 679 analyses — high quality)
        old_grade = df.loc[idx, 'Resource Grade (TGC%)']
        if pd.notna(nor_row['TC_pct']):
            if pd.isna(old_grade) or abs(old_grade - nor_row['TC_pct']) > 0.5:
                df.loc[idx, 'Resource Grade (TGC%)'] = nor_row['TC_pct']
                log_change(idx, existing_name, 'Resource Grade (TGC%)',
                           old_grade, nor_row['TC_pct'], SOURCE_NORWAY)
        
        # Update tonnage if missing
        old_resource = df.loc[idx, 'Total Resource (mt)']
        if pd.notna(nor_row['Tonnage_t']) and (pd.isna(old_resource) or old_resource == 0):
            df.loc[idx, 'Total Resource (mt)'] = nor_row['Tonnage_t']
            log_change(idx, existing_name, 'Total Resource (mt)',
                       old_resource, nor_row['Tonnage_t'], SOURCE_NORWAY)
        
        # Update contained graphite if missing
        old_graphite = df.loc[idx, 'Contained Graphite (mt)']
        if pd.notna(nor_row['Contained_Graphite_t']) and (pd.isna(old_graphite) or old_graphite == 0):
            df.loc[idx, 'Contained Graphite (mt)'] = nor_row['Contained_Graphite_t']
        
        # Add province info to location description
        if pd.isna(df.loc[idx, 'Location Description']):
            df.loc[idx, 'Location Description'] = f"Northern Norway, {nor_row['Province']} province"
        
        # Append source
        existing_source = str(df.loc[idx, 'Data Source']) if pd.notna(df.loc[idx, 'Data Source']) else ''
        if 'Gautneb' not in existing_source:
            df.loc[idx, 'Data Source'] = f"{existing_source}; {SOURCE_NORWAY}".strip('; ')
        
        updated_nor += 1
        print(f"  ✓ Updated: {nor_name} → {existing_name}")
    
    else:
        # ── Add new record ──
        new_row = {
            'Region': 'Europe',
            'Source_Layer': 'Norway_Gautneb_2020',
            'Country (Short Form)': 'Norway',
            'Name': nor_name,
            'Commodity': 'Graphite',
            'Status': 'Exploration',
            'Deposit Type': 'Flake graphite (granulite facies)',
            'ADM1 Name': nor_row['Province'],
            'Location Description': f"Northern Norway, {nor_row['Province']} province",
            'Data Source': SOURCE_NORWAY,
            'Reference Year': 2020,
            'Facility Comments': f"Occurrence #{nor_row['Occ_No']} in Gautneb et al. (2020). "
                                 f"Inferred resource from geophysics + sampling.",
            
            # Resource data
            'Total Resource (mt)': nor_row['Tonnage_t'],
            'Resource Grade (TGC%)': nor_row['TC_pct'],
            'Contained Graphite (mt)': nor_row['Contained_Graphite_t'],
            
            # These are all inferred resources
            'Inferred Resource (mt)': nor_row['Tonnage_t'],
            'Inferred Grade (TGC%)': nor_row['TC_pct'],
            'Inferred Graphite (mt)': nor_row['Contained_Graphite_t'],
        }
        
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
        log_change(len(df)-1, nor_name, 'NEW RECORD', None, nor_name, SOURCE_NORWAY)
        added_nor += 1
        print(f"  + Added: {nor_name} ({nor_row['Province']}, "
              f"{nor_row['TC_pct']}% TC, {nor_row['Tonnage_Mt']} Mt)")

print(f"\nUpdated {updated_nor} existing Norway records")
print(f"Added {added_nor} new Norway records")

Existing Norway records in dataset: 10
Names: ['Trælen', 'Vardfjellet', 'Grunnvåg', 'Vikeid Central', 'Smines', 'Haugsnes', 'Instøya', 'Romset', 'Litljkollen', 'Vikeid West']
  ✓ Updated: Trælen → Trælen
  ✓ Updated: Vardfjellet → Vardfjellet
  + Added: Hesten (Senja, 5.8% TC, 2.07 Mt)
  + Added: Bukken (Senja, 6.5% TC, 51.03 Mt)
  ✓ Updated: Litljkollen → Litljkollen
  ✓ Updated: Grunnvåg → Grunnvåg
  + Added: Skardsvåg (Senja, 2.1% TC, nan Mt)
  ✓ Updated: Haugsnes → Haugsnes
  + Added: Kjerkhaugen (Vesterålen, 6.5% TC, 2.42 Mt)
  + Added: Møkland (Vesterålen, 13.2% TC, 3.4 Mt)
  + Added: Rise (Vesterålen, 7.9% TC, 0.19 Mt)
  + Added: Sommarland (Vesterålen, 12.5% TC, 0.85 Mt)
  + Added: Brenna (Vesterålen, 10.1% TC, 7.94 Mt)
  + Added: Evassåsen (Vesterålen, 7.6% TC, 2.12 Mt)
  ✓ Updated: Vikeid Central → Vikeid Central
  ✓ Updated: Vikeid West → Vikeid Central
  + Added: Ånstad (Vesterålen, 36.8% TC, 0.21 Mt)
  + Added: Alsvåg (Vesterålen, 8.9% TC, 0.25 Mt)
  ✓ Updated: Instøya → I

In [119]:
# ── Norway summary ──
norway_all = df[df['Country (Short Form)'] == 'Norway']
norway_with_resource = norway_all[norway_all['Total Resource (mt)'].notna()]

print(f"\n{'='*70}")
print(f"NORWAY DATA SUMMARY")
print(f"{'='*70}")
print(f"Source: {SOURCE_NORWAY}")
print(f"Total Norway records: {len(norway_all)}")
print(f"  With resource data: {len(norway_with_resource)}")
print(f"  Total ore resource: {norway_with_resource['Total Resource (mt)'].sum()/1e6:.1f} Mt")
print(f"  Total contained graphite: {norway_with_resource['Contained Graphite (mt)'].sum()/1e6:.1f} Mt")
print(f"  Average grade: {norway_with_resource['Resource Grade (TGC%)'].mean():.1f}% TC")

print(f"\nBy province:")
for prov in ['Senja', 'Vesterålen', 'Holandsfjorden']:
    prov_mask = norway_all['Location Description'].str.contains(prov, na=False)
    prov_df = norway_all[prov_mask]
    prov_resource = prov_df[prov_df['Total Resource (mt)'].notna()]
    if not prov_resource.empty:
        print(f"  {prov}: {len(prov_df)} occurrences, "
              f"{prov_resource['Total Resource (mt)'].sum()/1e6:.1f} Mt ore, "
              f"{prov_resource['Contained Graphite (mt)'].sum()/1e6:.2f} Mt graphite, "
              f"avg {prov_resource['Resource Grade (TGC%)'].mean():.1f}% TC")

print(f"\nAll Norway deposits:")
print(norway_all[['Name', 'Total Resource (mt)', 'Resource Grade (TGC%)', 
                   'Contained Graphite (mt)', 'Source_Layer']].to_string(index=False))


NORWAY DATA SUMMARY
Source: Gautneb et al. (2020), Minerals 10(7), 626, Table 1. doi:10.3390/min10070626
Total Norway records: 28
  With resource data: 26
  Total ore resource: 241.7 Mt
  Total contained graphite: 21.2 Mt
  Average grade: 11.8% TC

By province:
  Senja: 7 occurrences, 125.0 Mt ore, 8.21 Mt graphite, avg 10.5% TC
  Vesterålen: 18 occurrences, 84.5 Mt ore, 9.38 Mt graphite, avg 12.8% TC
  Holandsfjorden: 2 occurrences, 2.5 Mt ore, 0.23 Mt graphite, avg 7.6% TC

All Norway deposits:
          Name  Total Resource (mt)  Resource Grade (TGC%)  Contained Graphite (mt)             Source_Layer
        Trælen            1800000.0                   31.0                 558000.0 USGS_Flake_Graphite_2024
   Vardfjellet           12840000.0                    9.2                1181280.0 USGS_Flake_Graphite_2024
      Grunnvåg           22770000.0                    5.2                1184040.0 USGS_Flake_Graphite_2024
Vikeid Central            8890000.0                   11.3   

In [120]:
df

,Region,Source_Layer,Country (Short Form),Facility Name,Site Name,Deposit Name,Commodity Type,Commodity,Commodity 1,Facility Status,...,Contained Graphite (mt),Measured Resource (mt),Measured Grade (TGC%),Measured Graphite (mt),Indicated Resource (mt),Indicated Grade (TGC%),Indicated Graphite (mt),Inferred Resource (mt),Inferred Grade (TGC%),Inferred Graphite (mt)
0,Africa,AFR_Mineral_Facilities,Madagascar,Gallois Mine,NaN,NaN,Industrial,Graphite,NaN,Assumed Active,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Africa,AFR_Mineral_Facilities,Madagascar,Graphmada Mine,NaN,NaN,Industrial,Graphite,NaN,Assumed Active,...,2785500.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Africa,AFR_Mineral_Facilities,Madagascar,Mine in Atsinanana Region,NaN,NaN,Industrial,Graphite,NaN,Assumed Active,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Africa,AFR_Mineral_Facilities,Madagascar,Sahamamy-Sahasoa Mine,NaN,NaN,Industrial,Graphite,NaN,Assumed Active,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Africa,AFR_Mineral_Facilities,Mozambique,Ancuabe Graphite Mine,NaN,NaN,Industrial,Graphite,NaN,Assumed Active,...,4584000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,Europe,Norway_Gautneb_2020,Norway,NaN,NaN,NaN,NaN,Graphite,NaN,NaN,...,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,20000.0,11.7,20000.0
260,Europe,Norway_Gautneb_2020,Norway,NaN,NaN,NaN,NaN,Graphite,NaN,NaN,...,350000.0,NaN,NaN,NaN,NaN,NaN,NaN,3660000.0,9.6,350000.0
261,Europe,Norway_Gautneb_2020,Norway,NaN,NaN,NaN,NaN,Graphite,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
262,Europe,Norway_Gautneb_2020,Norway,NaN,NaN,NaN,NaN,Graphite,NaN,NaN,...,20000.0,NaN,NaN,NaN,NaN,NaN,NaN,600000.0,4.1,20000.0


# 3. Deduplication

In [121]:
shared_capacity_duplicates = [
    ('Brazil', ['Itapecerica', 'Pedra Azul'], 'Salto da Divisa'),
    ('Mexico', ['San Juan Mine', 'Topiyeca Mine'], 'Lourdes Mine'),
]

total_removed = 0
for country, names_to_remove, keep_name in shared_capacity_duplicates:
    keep_mask = (df['Country (Short Form)'] == country) & (df['Name'] == keep_name)
    if keep_mask.sum() == 0:
        print(f"  ⚠ '{keep_name}' ({country}) not found — skipping")
        continue
    for dup_name in names_to_remove:
        dup_mask = (df['Country (Short Form)'] == country) & (df['Name'] == dup_name)
        if dup_mask.sum() == 0:
            continue
        dup_idx = df[dup_mask].index[0]
        log_change(dup_idx, dup_name, 'REMOVED (shared capacity)',
                   dup_name, f'Duplicate of {keep_name}', 'Deduplication')
        df = df.drop(index=dup_idx).reset_index(drop=True)
        total_removed += 1
        print(f"  ✗ Removed: {dup_name}")

print(f"Removed {total_removed} duplicates")

  ✗ Removed: Itapecerica
  ✗ Removed: Pedra Azul
  ✗ Removed: San Juan Mine
  ✗ Removed: Topiyeca Mine
Removed 4 duplicates


In [122]:
# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4: FINAL CLEANUP
# ══════════════════════════════════════════════════════════════════════════════

# ── 4.1 Drop all unnecessary columns ──
cols_to_drop = [
    # Original source columns (merged in Section 1)
    'Facility Name', 'Site Name', 'Deposit Name',
    'Facility Status', 'Exploration Status', 'Operating Status', 'Development Status',
    'Facility Type', 'Commodity 1', 'Commodity Type',
    'Owner Name', 'Equity Owner 1', 'AMD1 Name',
    'Year', 'Source Year',
    'Location Confidence', 'Location Notes', 'Alternate Name',
    'Mineral Facility Data Source',
    # Metadata/source columns
    'USGS Facility ID', 'Label', 'Multiple Commodities', 'Multiple Products',
    'Location Description', 'Location Accuracy', 'GIS Data Source',
    'Equity Owner 2', 'Equity Owner 3', 'Equity Owner 4', 'Equity Owner 5',
    'Equity Owner 6', 'Data Source', 'Alternate Name(s)',
    'Deposit ID', 'USGS MRData ID', 'Exploration ID', 'Location Notes',
    'USGS WMED ID', 'Owner Type', 'Owner Share',
    'Location Source 1', 'Location Source 2',
    'Operation Source 1', 'Operation Source 2',
    'Company Source 1', 'Company Source 2',
    'Commodity Source 1', 'Commodity Source 2',
    'DsgAttr9', 'Capacity Source 1', 'Capacity Source 2',
    'Other Source', 'Source notes', 'Development ID',
    'Operating Stage Source 1', 'Operating Stage Source 2',
    'Commodity 9', 'Commodity 10', 'Commodity 11', 'Critical Commodity',
    'ROWID', 'ISO_CC', 'FACID_NUM', 'Estimated Capacity', 'Source ID',
    'Capacity Notes', 'Shared Capacity', 'Product Notes',
    'Commodity 2', 'Commodity 3', 'Commodity 4', 'Commodity 5',
    'Commodity 6', 'Commodity 7', 'Commodity 8',
    'Facility Comments', 'geometry',
]
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

# ── 4.2 Decode Feature Type short codes ──
df['Feature Type'] = df['Feature Type'].replace({
    'B': 'Brine operation', 'M': 'Mines and Quarries',
    'MP': 'Mineral Processing Plants', 'P': 'Plant',
    'OG': 'Oil and gas fields', 'OR': 'Oil and gas refineries',
    'Mine': 'Mines and Quarries', 'Mine and plant': 'Mineral Processing Plants',
})

# Remove any remaining Mineral Processing Plants
df = df[df['Feature Type'] != 'Mineral Processing Plants']

# ── 4.3 Remove non-graphite records ──
df = df[df['Commodity'] == 'Graphite'].reset_index(drop=True)

# ── 4.3b Remove Synthetic Graphite ──
synth_mask = df['Commodity Product'] == 'Synthetic Graphite'
print(f"Removing {synth_mask.sum()} Synthetic Graphite row(s)")
df = df[~synth_mask].reset_index(drop=True)

# ── 4.3c Set Status for deposit records ──
dep_mask = df['Source_Layer'].str.contains('Mineral_Deposits', case=False, na=False)
changed = (dep_mask & (df['Status'] != 'Exploration')).sum()
df.loc[dep_mask, 'Status'] = 'Exploration'
print(f"Set Status → 'Exploration' for {changed} Mineral_Deposits rows")

# ── 4.4 Standardise Status ──
df['Status'] = df['Status'].replace({
    'Assumed Active': 'Production', 'A': 'Production',
    'C&M': 'Care and Maintenance',
    'Feasibility Study': 'Feasibility', 'Preliminary Study': 'Feasibility',
    'Pre-Production': 'Development', 'Pre-production': 'Development',
    'Construction': 'Development',
    'Reserves Development': 'Development', 'Reserves development': 'Development',
})

# ── 4.5 Final column order ──
col_order = [
    'Region', 'Source_Layer', 'Country (Short Form)', 'Name',
    'Commodity', 'Deposit Type', 'Commodity Product',
    'Status', 'Feature Type',
    'ADM1 Name', 'DD Latitude', 'DD Longitude',
    'Annual Production Capacity', 'Capacity Units',
    'Major Operating Company', 'Owner',
    'Reference Year',
    'Total Resource (mt)', 'Resource Grade (TGC%)', 'Contained Graphite (mt)',
    'Measured Resource (mt)', 'Measured Grade (TGC%)', 'Measured Graphite (mt)',
    'Indicated Resource (mt)', 'Indicated Grade (TGC%)', 'Indicated Graphite (mt)',
    'Inferred Resource (mt)', 'Inferred Grade (TGC%)', 'Inferred Graphite (mt)',
]
ordered = [c for c in col_order if c in df.columns]
remaining = [c for c in df.columns if c not in ordered]
df = df[ordered + remaining]

print(f"Final dataset: {len(df)} rows, {len(df.columns)} columns")
print(f"\nStatus: {df['Status'].value_counts(dropna=False).to_string()}")
print(f"\nFeature Type: {df['Feature Type'].value_counts(dropna=False).to_string()}")

Removing 0 Synthetic Graphite row(s)
Set Status → 'Exploration' for 52 Mineral_Deposits rows
Final dataset: 248 rows, 30 columns

Status: Status
Exploration             177
Production               28
Inactive                 16
Feasibility              13
Development              11
Care and Maintenance      3

Feature Type: Feature Type
NaN                   218
Mines and Quarries     30


In [123]:
# ── SAVE ──
df.to_csv('./outputs/Global_Graphite_Projects_final.csv', index=False)
print(f"Saved: {len(df)} rows to Global_Graphite_Projects_final.csv")

pd.DataFrame(change_log).to_csv('./outputs/change_log.csv', index=False)
print(f"Saved: {len(change_log)} changes to change_log.csv")

Saved: 248 rows to Global_Graphite_Projects_final.csv
Saved: 133 changes to change_log.csv
